In [2]:
from pycromanager import Core
import time
from pytz import timezone
from datetime import datetime
import cv2
from pycromanager import Acquisition, multi_d_acquisition_events
import numpy as np
import os
import json
import pandas as pd
from pycromanager import Core
import skimage
import warnings

warnings.filterwarnings("ignore")
import shutil
import copy
import math
import re
from PIL import Image, ImageSequence

class scope_constant():
    piezo_focus_start_pos = -15
    piezo_focus_end_pos = 15
    piezo_step = 1.5

    piezo_maxpro_start_pos = -15
    piezo_maxpro_end_pos = 15

    ZDrive_safe_pos = 190.86
    XYStage_fluidics_safe_pos = [-56900.7, -3062.1]
    XYStage_image_safe_pos = [44198.6, -4834.5]

    # scope_geneseq_first_channel=["G","T","A","C","DIC"]
    # scope_geneseq_subchannel=["G","T","A","C"]
    # scope_geneseq_hybchannel=["GFP", "T", "TxRed", "A", "DAPI", "DIC"]
    scope_exposure_time_dict = {"G": 220, "T": 150, "A": 100, "C": 100, "DIC": 10, "GFP": 100, "TxRed": 100, "DAPI": 50}
    stack_num_focus = round(abs(piezo_focus_end_pos - piezo_focus_start_pos) / piezo_step)
    stack_num_maxpro = round(abs(piezo_maxpro_start_pos - piezo_maxpro_end_pos) / piezo_step)

    sharpen1 = np.array(([0, 1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]), dtype="int")
    stage_x_dir = -1;  # -1: left is larger
    stage_y_dir = 1;  # 1: bottom is larger
    pos_per_slice = 4;
    pixelsize = 0.33
    imwidth = 2048
    overlap = 10
def get_time():
    time_now = timezone('US/Pacific')
    time = str(datetime.now(time_now))[0:19] + "\n"
    return time




In [8]:
df=pd.read_csv(os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo","tiledregoffsetgeneseq01.csv"))

In [9]:
maxprojection_ls=df['switched_Posinfo']

In [10]:
img=Image.open(os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo","geneseq01","geneseq01_1","geneseq01_NDTiffStack.tif"))

In [11]:
img.close()

In [12]:

index=0
file="geneseq01_NDTiffStack.tif"
metadate_df=pd.DataFrame(columns=['img_name','img_index','channel','z','position'])
for i, page in enumerate(ImageSequence.Iterator(img)):
    img_dict=json.loads(page.tag_v2.get(51123)).get('Axes') #in matedata the key is 51123
    df = {'img_name':file , 'img_index': index, 'channel': img_dict.get('channel'),'z':img_dict.get('z'),'position':img_dict.get('Pos')}
    index=index+1
    metadate_df = metadate_df.append(df, ignore_index = True)

ValueError: Operation on closed image

In [ ]:
metadate_df

In [ ]:
def list_file(directory,substring):
    dir_list = os.listdir(directory)
    ls=[a for a in dir_list if substring in a]
    return ls

In [ ]:
max_ls=list_file(os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo","geneseq01","geneseq01_1"),'.tif')

In [ ]:
metadate_df=pd.DataFrame(columns=['img_name','img_index','channel','z','position'])
for file in max_ls:
    img=Image.open(os.path.join(os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo","geneseq01","geneseq01_1"),file))
    index=0
    for i, page in enumerate(ImageSequence.Iterator(img)):
        img_dict=json.loads(page.tag_v2.get(51123)).get('Axes') #in matedata the key is 51123
        df = {'img_name':file , 'img_index': index, 'channel': img_dict.get('channel'),'z':img_dict.get('z'),'position':img_dict.get('position')}
        index=index+1
        metadate_df = metadate_df.append(df, ignore_index = True)

In [4]:
def decode_NDtiff(pos):
    with open(os.path.join(pos_path,cycle,cycle+"_1","NDTiff.index"), 'rb') as f:
        bytes = f.read()
        f.close()
    s = bytes.decode('ansi')
    l1 = re.findall(r'(?<={}).*?(?={})'.format('\{"Pos":"', "\}"), s)
    l2 = re.findall(r'(?<={}).*?(?={})'.format(cycle, ".tif"), s)
    result = [json.loads('{"Pos":"' + i + "}") for i in l1]
    pos_list=[i['Pos'] for i in result]
    index=[]
    for i in range(len(pos_list)):
        if pos_list[i]==pos:
            index.append(i)
    return l1,l2,index

def make_matedf(img_stack):
    metadate_df=pd.DataFrame(columns=['img_name','img_index','channel','z','position'])
    for file in img_stack:
        img=Image.open(os.path.join(pos_path,cycle,cycle+"_1",file))
        for i, page in enumerate(ImageSequence.Iterator(img)):
            img_dict=json.loads(page.tag_v2.get(51123)).get('Axes') #in matedata the key is 51123
            df = {'img_name':file , 'img_index': i, 'channel': img_dict.get('channel'),'z':img_dict.get('z'),'position':img_dict.get('Pos')}
            metadate_df = metadate_df.append(df, ignore_index = True)
    return metadate_df
def load_image(img):
    im = skimage.io.imread(os.path.join(pos_path,cycle,cycle+"_1",img))
    im=im.reshape(-1, *im.shape[-2:])
    return im

def list_file(directory,substring):
    dir_list = os.listdir(directory)
    ls=[a for a in dir_list if substring in a]
    int()
    return ls
def assign_stack_number(a):
    result=re.search('_NDTiffStack(.*).tif', a)
    index=result.group(1)
    if index=='':
        number=0
    else:
        index=index.replace('_','')
        number=int(index)
    return number
def find_pos(a):
     result=re.search('MAX_(.*)_', a)
     index=result.group(1)
     return index
    

In [108]:
finish_stack=[]

In [33]:
cycle="geneseq03"
pos="Pos1"
channel_ls=["G", "T", "A", "C"]
finish_stack=[]
pos_path=os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo")

In [34]:
if not os.path.exists(os.path.join(pos_path,cycle,'maxprojection')):
    os.mkdir(os.path.join(pos_path,cycle,'maxprojection'))

In [3]:
cycle="geneseq03"
channel_ls=["G", "T", "A", "C"]
finish_stack=[]
pos_path=os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo")
if not os.path.exists(os.path.join(pos_path,cycle,'maxprojection')):
    os.mkdir(os.path.join(pos_path,cycle,'maxprojection'))
file=list_file(os.path.join(pos_path,cycle,cycle+"_1"),".tif")
left_image=pd.DataFrame(columns=['image_name','image_index','channel','z','position'])
if len(file)>1:
    sort_list=[assign_stack_number(f) for f in file]
    sort_file = [x for _,x in sorted(zip(sort_list,file))]   
    max_file = [x for x in sort_file if x not in finish_stack]
    for f in max_file:
        df=make_matedf([f])
        if len(left_image)!=0:
            df=pd.concat([df,left_image],axis=0)
        img=skimage.io.imread(os.path.join(pos_path,cycle,cycle+"_1",f))
        df_group=df.groupby(['position','channel']).size().reset_index(name='count')
        for index, row in df_group.iterrows():
            if row['count']==20:
                img_i=df[(df['position']==row['position'] )& (df['channel']==row['channel'])]['img_index']
                max_projection=[]
                for i in img_i:
                    max_projection.append(img[i])
                max=np.max(np.array(max_projection), axis=(0))
                max_name='MAX_'+row['position']+'_'+row['channel']+'.tif'
                skimage.io.imsave(os.path.join(pos_path, cycle, max_name), max, photometric='minisblack')
            else:
                left_image=df[(df['position'] == row['position']) & (df['channel'] == row['channel'])]
        finish_stack.append(f)
    img_file=list_file(os.path.join(pos_path,cycle),".tif")
    a=[find_pos(i) for i in img_file]
    a_unique=list(set(a))
    for i in a_unique:
        n=a.count(i)
        if n==len(channel_ls):
            max_img=[]
            for ch in channel_ls:
                img_max=skimage.io.imread(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
                print('MAX_'+i+'_'+ch+'.tif')
                max_img.append(img_max)
                os.remove(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
                print('remove'+'MAX_'+i+'_'+ch+'.tif')
            skimage.io.imsave(os.path.join(pos_path, cycle,'maxprojection','MAX_'+i+'.tif'), np.array(max_img), photometric='minisblack')    
            
else:
    pass

MAX_Pos163_G.tif
removeMAX_Pos163_G.tif
MAX_Pos163_T.tif
removeMAX_Pos163_T.tif
MAX_Pos163_A.tif
removeMAX_Pos163_A.tif
MAX_Pos163_C.tif
removeMAX_Pos163_C.tif
MAX_Pos136_G.tif
removeMAX_Pos136_G.tif
MAX_Pos136_T.tif
removeMAX_Pos136_T.tif
MAX_Pos136_A.tif
removeMAX_Pos136_A.tif
MAX_Pos136_C.tif
removeMAX_Pos136_C.tif
MAX_Pos169_G.tif
removeMAX_Pos169_G.tif
MAX_Pos169_T.tif
removeMAX_Pos169_T.tif
MAX_Pos169_A.tif
removeMAX_Pos169_A.tif
MAX_Pos169_C.tif
removeMAX_Pos169_C.tif
MAX_Pos125_G.tif
removeMAX_Pos125_G.tif
MAX_Pos125_T.tif
removeMAX_Pos125_T.tif
MAX_Pos125_A.tif
removeMAX_Pos125_A.tif
MAX_Pos125_C.tif
removeMAX_Pos125_C.tif
MAX_Pos63_G.tif
removeMAX_Pos63_G.tif
MAX_Pos63_T.tif
removeMAX_Pos63_T.tif
MAX_Pos63_A.tif
removeMAX_Pos63_A.tif
MAX_Pos63_C.tif
removeMAX_Pos63_C.tif
MAX_Pos51_G.tif
removeMAX_Pos51_G.tif
MAX_Pos51_T.tif
removeMAX_Pos51_T.tif
MAX_Pos51_A.tif
removeMAX_Pos51_A.tif
MAX_Pos51_C.tif
removeMAX_Pos51_C.tif
MAX_Pos130_G.tif
removeMAX_Pos130_G.tif
MAX_Pos130_T.tif

In [6]:
df

,img_name,img_index,channel,z,position
0,geneseq03_NDTiffStack_30.tif,0,A,10,Pos192
1,geneseq03_NDTiffStack_30.tif,1,A,11,Pos192
2,geneseq03_NDTiffStack_30.tif,2,A,12,Pos192
3,geneseq03_NDTiffStack_30.tif,3,A,13,Pos192
4,geneseq03_NDTiffStack_30.tif,4,A,14,Pos192
...,...,...,...,...,...
505,geneseq03_NDTiffStack_30.tif,505,C,15,Pos198
506,geneseq03_NDTiffStack_30.tif,506,C,16,Pos198
507,geneseq03_NDTiffStack_30.tif,507,C,17,Pos198
508,geneseq03_NDTiffStack_30.tif,508,C,18,Pos198


In [5]:
df=make_matedf(['geneseq03_NDTiffStack_30.tif'])
img=skimage.io.imread(os.path.join(pos_path,cycle,cycle+"_1",max_file[0]))
df_group=df.groupby(['position','channel']).size().reset_index(name='count')
# for index, row in df_group.iterrows():
#     if row['count']==20:
#         img_i=df[(df['position']==row['position'] )& (df['channel']==row['channel'])]['img_index']
#         max_projection=[]
#         for i in img_i:
#             max_projection.append(img[i])
#         max=np.max(np.array(max_projection), axis=(0))
#         max_name='MAX_'+row['position']+'_'+row['channel']+'.tif'
#         skimage.io.imsave(os.path.join(pos_path, cycle, max_name), max, photometric='minisblack')

In [28]:
img_file=list_file(os.path.join(pos_path,cycle),".tif")
a=[find_pos(i) for i in img_file]
a_unique=list(set(a))
for i in a_unique:
    n=a.count(i)
    if n==len(channel_ls):
        max_img=[]
        for ch in channel_ls:
            img_max=skimage.io.imread(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
            print('MAX_'+i+'_'+ch+'.tif')
            max_img.append(img_max)
            os.remove(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
            print('remove'+'MAX_'+i+'_'+ch+'.tif')
        skimage.io.imsave(os.path.join(pos_path, cycle,'maxprojection','MAX_'+i+'.tif'), np.array(max_img), photometric='minisblack')    

MAX_Pos13_G.tif
removeMAX_Pos13_G.tif
MAX_Pos13_T.tif
removeMAX_Pos13_T.tif
MAX_Pos13_A.tif
removeMAX_Pos13_A.tif
MAX_Pos13_C.tif
removeMAX_Pos13_C.tif
MAX_Pos18_G.tif
removeMAX_Pos18_G.tif
MAX_Pos18_T.tif
removeMAX_Pos18_T.tif
MAX_Pos18_A.tif
removeMAX_Pos18_A.tif
MAX_Pos18_C.tif
removeMAX_Pos18_C.tif
MAX_Pos19_G.tif
removeMAX_Pos19_G.tif
MAX_Pos19_T.tif
removeMAX_Pos19_T.tif
MAX_Pos19_A.tif
removeMAX_Pos19_A.tif
MAX_Pos19_C.tif
removeMAX_Pos19_C.tif
MAX_Pos17_G.tif
removeMAX_Pos17_G.tif
MAX_Pos17_T.tif
removeMAX_Pos17_T.tif
MAX_Pos17_A.tif
removeMAX_Pos17_A.tif
MAX_Pos17_C.tif
removeMAX_Pos17_C.tif
MAX_Pos14_G.tif
removeMAX_Pos14_G.tif
MAX_Pos14_T.tif
removeMAX_Pos14_T.tif
MAX_Pos14_A.tif
removeMAX_Pos14_A.tif
MAX_Pos14_C.tif
removeMAX_Pos14_C.tif
MAX_Pos15_G.tif
removeMAX_Pos15_G.tif
MAX_Pos15_T.tif
removeMAX_Pos15_T.tif
MAX_Pos15_A.tif
removeMAX_Pos15_A.tif
MAX_Pos15_C.tif
removeMAX_Pos15_C.tif
MAX_Pos16_G.tif
removeMAX_Pos16_G.tif
MAX_Pos16_T.tif
removeMAX_Pos16_T.tif
MAX_Pos16_A.

In [115]:
img_file=list_file(os.path.join(pos_path,cycle),".tif")
img_file

['MAX_Pos1.tif',
 'MAX_Pos10_A.tif',
 'MAX_Pos10_C.tif',
 'MAX_Pos10_G.tif',
 'MAX_Pos10_T.tif',
 'MAX_Pos11_A.tif',
 'MAX_Pos11_C.tif',
 'MAX_Pos11_G.tif',
 'MAX_Pos11_T.tif',
 'MAX_Pos12_A.tif',
 'MAX_Pos12_C.tif',
 'MAX_Pos12_G.tif',
 'MAX_Pos12_T.tif',
 'MAX_Pos13_A.tif',
 'MAX_Pos13_G.tif',
 'MAX_Pos13_T.tif',
 'MAX_Pos2.tif',
 'MAX_Pos3.tif',
 'MAX_Pos4.tif',
 'MAX_Pos5.tif',
 'MAX_Pos6.tif',
 'MAX_Pos7_A.tif',
 'MAX_Pos7_C.tif',
 'MAX_Pos7_G.tif',
 'MAX_Pos8_A.tif',
 'MAX_Pos8_C.tif',
 'MAX_Pos8_G.tif',
 'MAX_Pos8_T.tif',
 'MAX_Pos9_A.tif',
 'MAX_Pos9_C.tif',
 'MAX_Pos9_G.tif',
 'MAX_Pos9_T.tif']

In [109]:
finish_stack.append(max_file[0])

In [110]:
finish_stack

['geneseq02_NDTiffStack.tif']

In [111]:
sort_list=[assign_stack_number(f) for f in file]
sort_file = [x for _,x in sorted(zip(sort_list,file))]   
max_file = [x for x in sort_file if x not in finish_stack]

In [112]:
max_file

['geneseq02_NDTiffStack_1.tif',
 'geneseq02_NDTiffStack_2.tif',
 'geneseq02_NDTiffStack_3.tif',
 'geneseq02_NDTiffStack_4.tif',
 'geneseq02_NDTiffStack_5.tif',
 'geneseq02_NDTiffStack_6.tif',
 'geneseq02_NDTiffStack_7.tif',
 'geneseq02_NDTiffStack_8.tif',
 'geneseq02_NDTiffStack_9.tif',
 'geneseq02_NDTiffStack_10.tif',
 'geneseq02_NDTiffStack_11.tif',
 'geneseq02_NDTiffStack_12.tif',
 'geneseq02_NDTiffStack_13.tif',
 'geneseq02_NDTiffStack_14.tif',
 'geneseq02_NDTiffStack_15.tif',
 'geneseq02_NDTiffStack_16.tif',
 'geneseq02_NDTiffStack_17.tif',
 'geneseq02_NDTiffStack_18.tif',
 'geneseq02_NDTiffStack_19.tif',
 'geneseq02_NDTiffStack_20.tif',
 'geneseq02_NDTiffStack_21.tif',
 'geneseq02_NDTiffStack_22.tif',
 'geneseq02_NDTiffStack_23.tif',
 'geneseq02_NDTiffStack_24.tif',
 'geneseq02_NDTiffStack_25.tif',
 'geneseq02_NDTiffStack_26.tif',
 'geneseq02_NDTiffStack_27.tif',
 'geneseq02_NDTiffStack_28.tif',
 'geneseq02_NDTiffStack_29.tif',
 'geneseq02_NDTiffStack_30.tif',
 'geneseq02_NDTiffS

In [102]:
df=make_matedf([max_file[0]])
img=skimage.io.imread(os.path.join(pos_path,cycle,cycle+"_1",max_file[0]))
df_group=df.groupby(['position','channel']).size().reset_index(name='count')
for index, row in df_group.iterrows():
    if row['count']==20:
        img_i=df[(df['position']==row['position'] )& (df['channel']==row['channel'])]['img_index']
        max_projection=[]
        for i in img_i:
            max_projection.append(img[i])
        max=np.max(np.array(max_projection), axis=(0))
        max_name='MAX_'+row['position']+'_'+row['channel']+'.tif'
        skimage.io.imsave(os.path.join(pos_path, cycle, max_name), max, photometric='minisblack')
img_file=list_file(os.path.join(pos_path,cycle),".tif")
a=[find_pos(i) for i in img_file]
a_unique=list(set(a))
for i in a_unique:
    n=a.count(i)
    if n==len(channel_ls):
        max_img=[]
        for ch in channel_ls:
            img_max=skimage.io.imread(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
            print('MAX_'+i+'_'+ch+'.tif')
            max_img.append(img_max)
            os.remove(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
        skimage.io.imsave(os.path.join(pos_path, cycle,'MAX_'+i+'.tif'), np.array(max_img), photometric='minisblack')
        
            
            
            
    




finish_stack.append(max_file[0])

In [103]:
a_unique

['Pos5', 'Pos2', 'Pos6', 'Pos4', 'Pos3', 'Pos1', 'Pos7']

In [104]:

for i in a_unique:
    n=a.count(i)
    if n==len(channel_ls):
        max_img=[]
        for ch in channel_ls:
            img_max=skimage.io.imread(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
            print('MAX_'+i+'_'+ch+'.tif')
            max_img.append(img_max)
            os.remove(os.path.join(pos_path,cycle,'MAX_'+i+'_'+ch+'.tif'))
        skimage.io.imsave(os.path.join(pos_path, cycle,'MAX_'+i+'.tif'), np.array(max_img), photometric='minisblack')


MAX_Pos5_G.tif
MAX_Pos5_T.tif
MAX_Pos5_A.tif
MAX_Pos5_C.tif
MAX_Pos2_G.tif
MAX_Pos2_T.tif
MAX_Pos2_A.tif
MAX_Pos2_C.tif
MAX_Pos6_G.tif
MAX_Pos6_T.tif
MAX_Pos6_A.tif
MAX_Pos6_C.tif
MAX_Pos4_G.tif
MAX_Pos4_T.tif
MAX_Pos4_A.tif
MAX_Pos4_C.tif
MAX_Pos3_G.tif
MAX_Pos3_T.tif
MAX_Pos3_A.tif
MAX_Pos3_C.tif
MAX_Pos1_G.tif
MAX_Pos1_T.tif
MAX_Pos1_A.tif
MAX_Pos1_C.tif


In [94]:
a_unique

['Pos5', 'Pos2', 'Pos6', 'Pos4', 'Pos3', 'Pos1', 'Pos7']

In [76]:
df_group=df.groupby(['position','channel']).size().reset_index(name='count')
for index, row in df_group.iterrows():
    if row['count']==20:
        img_i=df[(df['position']==row['position'] )& (df['channel']==row['channel'])]['img_index']
        max_projection=[]
        for i in img_i:
            max_projection.append(img[i])
        max=np.max(np.array(max_projection), axis=(0))
        max_name='MAX_'+row['position']+'_'+row['channel']+'.tif'
        skimage.io.imsave(os.path.join(pos_path, cycle, max_name), max, photometric='minisblack')
        

In [78]:
df_group

,position,channel,count
0,Pos1,A,20
1,Pos1,C,20
2,Pos1,G,20
3,Pos1,T,20
4,Pos2,A,20
5,Pos2,C,20
6,Pos2,G,20
7,Pos2,T,20
8,Pos3,A,20
9,Pos3,C,20


In [75]:
df_group

,position,channel,count
0,Pos1,A,20
1,Pos1,C,20
2,Pos1,G,20
3,Pos1,T,20
4,Pos2,A,20
5,Pos2,C,20
6,Pos2,G,20
7,Pos2,T,20
8,Pos3,A,20
9,Pos3,C,20


In [ ]:
def assign_stack_number(a):
    result=re.search('_NDTiffStack(.*).tif', a)
    index=result.group(1)
    if index=='':
        number=0
    else:
        index=index.replace('_','')
        number=int(index)
    return number
        
        
    
    

In [34]:
a=file[1]
a

'geneseq02_NDTiffStack_1.tif'

In [38]:
result=re.search('_NDTiffStack(.*).tif', a)
index=result.group(1)
index.replace('_','')

'1'

In [8]:
cycle="geneseq01"
pos="Pos1"
channel_ls=["G", "T", "A", "C", "DIC"]
maxprojection_stack=100
pos_path=os.path.join("D:\\","20240104_fullcycle_withhyb_withpiezo")
l1,l2,index=decode_NDtiff(pos)
while len(index)!=maxprojection_stack:
    time.sleep(3)
    l1,l2,index=decode_NDtiff(pos)
    print("not enough pic")
tiff_name=list(set([cycle+l2[i]+'.tif' for i in index]))
metadate_df=make_matedf(tiff_name)
max_name="MAX_"+pos+'.tif'
maxprojection=[]
img_stack_name_list=pd.unique(metadate_df['img_name'])
img_stack=[load_image(name) for name in img_stack_name_list]
img_dict = dict(zip(img_stack_name_list, img_stack))  
for ch in channel_ls:
    img_stack_subdf=metadate_df[(metadate_df['channel']==ch) &(metadate_df['position']==pos)]  
    img_stack_name_list=pd.unique(metadate_df['img_name'])
    im_extract_all=[]
    for name in pd.unique(img_stack_name_list):
        index=img_stack_subdf[img_stack_subdf['img_name']==name]['img_index']
        for i in index:
            im_extract_all.append(img_dict[name][i])
        channel_max=np.max(np.array(im_extract_all), axis=(0))
    maxprojection.append(channel_max)
skimage.io.imsave(os.path.join(pos_path,cycle,max_name), np.array(maxprojection),photometric='minisblack')    

In [7]:
a='MAX_Pos1_C.tif'

In [11]:
result=re.search('MAX_(.*)_', a)

In [12]:
result.group(1)

'Pos1'

In [13]:
img_file =list_file(os.path.join(pos_path, cycle), ".tif")

In [15]:
a='geneseq03_NDTiffStack.tif'

In [17]:
cycle='geneseq03'

In [19]:
df=pd.read_csv(os.path.join(pos_path,cycle,"geneseq03_NDTiffStack_3.csv"))

In [29]:
pd.unique(df['img_name']).tolist()

['geneseq03_NDTiffStack_3.tif', 'geneseq03_NDTiffStack_2.tif']

In [21]:
df_group = df.groupby(['position', 'channel']).size().reset_index(name='count')

In [31]:
df[(df['position'] == 'Pos20') & (df['channel'] == 'G')]

,Unnamed: 0,img_name,img_index,channel,z,position
0,0,geneseq03_NDTiffStack_3.tif,0,G,13,Pos20
1,1,geneseq03_NDTiffStack_3.tif,1,G,14,Pos20
2,2,geneseq03_NDTiffStack_3.tif,2,G,15,Pos20
3,3,geneseq03_NDTiffStack_3.tif,3,G,16,Pos20
4,4,geneseq03_NDTiffStack_3.tif,4,G,17,Pos20
5,5,geneseq03_NDTiffStack_3.tif,5,G,18,Pos20
6,6,geneseq03_NDTiffStack_3.tif,6,G,19,Pos20
387,498,geneseq03_NDTiffStack_2.tif,498,G,0,Pos20
388,499,geneseq03_NDTiffStack_2.tif,499,G,1,Pos20
389,500,geneseq03_NDTiffStack_2.tif,500,G,2,Pos20


In [32]:
from pycromanager import Acquisition, multi_d_acquisition_events
events = multi_d_acquisition_events(num_time_points=50)

In [ ]:
def get_time():
    time_now = timezone('US/Pacific')
    time = str(datetime.now(time_now))[0:19] + "\n"
    return time


In [33]:
events

[{'axes': {'time': 0}},
 {'axes': {'time': 1}},
 {'axes': {'time': 2}},
 {'axes': {'time': 3}},
 {'axes': {'time': 4}},
 {'axes': {'time': 5}},
 {'axes': {'time': 6}},
 {'axes': {'time': 7}},
 {'axes': {'time': 8}},
 {'axes': {'time': 9}},
 {'axes': {'time': 10}},
 {'axes': {'time': 11}},
 {'axes': {'time': 12}},
 {'axes': {'time': 13}},
 {'axes': {'time': 14}},
 {'axes': {'time': 15}},
 {'axes': {'time': 16}},
 {'axes': {'time': 17}},
 {'axes': {'time': 18}},
 {'axes': {'time': 19}},
 {'axes': {'time': 20}},
 {'axes': {'time': 21}},
 {'axes': {'time': 22}},
 {'axes': {'time': 23}},
 {'axes': {'time': 24}},
 {'axes': {'time': 25}},
 {'axes': {'time': 26}},
 {'axes': {'time': 27}},
 {'axes': {'time': 28}},
 {'axes': {'time': 29}},
 {'axes': {'time': 30}},
 {'axes': {'time': 31}},
 {'axes': {'time': 32}},
 {'axes': {'time': 33}},
 {'axes': {'time': 34}},
 {'axes': {'time': 35}},
 {'axes': {'time': 36}},
 {'axes': {'time': 37}},
 {'axes': {'time': 38}},
 {'axes': {'time': 39}},
 {'axes': 

In [41]:
print('a')
df=pd.read_csv(os.path.join(pos_path,cycle,"geneseq03_NDTiffStack_3.csv"))
acq=Acquisition(directory=os.path.join(pos_path), name='acquisition_name')
with Acquisition(directory=os.path.join(pos_path), name='acquisition_name') as acq:
    events = multi_d_acquisition_events(num_time_points=50)
    acq.acquire(events)
    print('b')
    acq.acquire(events)
    print('c')
print('b')
acq.mark_finished()

a
b
c
b


In [6]:

piezo_event_pos = np.arange(200 - 15, 200 +15,1.5)
def create_piezo_event(channel, pos,piezo_event_pos):
    piezo_event = []
    for c in channel:
        for z in range(0, len(piezo_event_pos)):
            piezo_event.append({'axes': {'Pos': pos, 'channel': c, 'z': z},
                                'config_group': ['Channels', c],
                                'exposure': scope_constant.scope_exposure_time_dict.get(c),
                                'z': piezo_event_pos[z]})
    return piezo_event

In [7]:
pos_path="D:\\20240104_fullcycle_withhyb_withpiezo"
cycle='geneseq03'
channel_list = ["G", "T", "A", "C"]
channel_Intensity = [220, 150, 100, 100]
maxproject_list=pd.read_csv(os.path.join(pos_path,"tiledregoffsetgeneseq03.csv"))
events = create_piezo_event(channel_list,'pos1',piezo_event_pos)

In [8]:
pos_path="D:\\20240104_fullcycle_withhyb_withpiezo"
cycle='geneseq01'
pos='Pos1_000_000'
img_meta=Image.open(os.path.join(pos_path, cycle, pos + "_1", pos+'_NDTiffStack.tif'))
img=skimage.io.imread(os.path.join(pos_path, cycle, pos + "_1", pos+'_NDTiffStack.tif'))
metadata_df=pd.DataFrame(columns=['img_index','channel','z','position'])
for i, page in enumerate(ImageSequence.Iterator(img_meta)):
        img_dict=json.loads(page.tag_v2.get(51123)).get('Axes') #in matedata the key is 51123
        df = {'img_index': i, 'channel': img_dict.get('channel'),'z':img_dict.get('z'),'position':img_dict.get('Pos')}
        metadata_df = metadata_df.append(df, ignore_index = True)
maxprojection=[]
for ch in channel_list:
    metadata_df[(metadata_df['channel'] == ch)]['img_index'].tolist()
    subimg=img[metadata_df[(metadata_df['channel'] == ch)]['img_index'].tolist(),:,:]
    maxprojection.append(np.max(np.array(subimg), axis=(0)))
skimage.io.imsave(os.path.join(self.pos_path, self.cycle, 'MAX_' + pos + '.tif'),np.array(maxprojection), photometric='minisblack')
    
    
    
    

array([[141, 154, 158, ..., 148, 116, 161],
       [152, 129, 159, ..., 173, 153, 149],
       [151, 153, 172, ..., 172, 143, 137],
       ...,
       [190, 237, 217, ..., 249, 271, 257],
       [189, 246, 247, ..., 208, 221, 232],
       [235, 217, 194, ..., 259, 253, 242]], dtype=uint16)